# ViT, byte tracking, EfficientNet

- ??: ViT, EfficientNet, RF-DETR? ByteTrack? ??? ???? ???? ??? ???? ?? ??.
- ????(??): ???
- ?? ?? ??: 12
- ?? ??: https://app.notion.com/p/3a2bf34cc2ef808ba630d6a536be36f2

> ??? ?? ? ????? ??? ????? ??????. ??? ? ??? ??? ??? ? ?? ?????.


In [ ]:
!pip install -q supervision
!pip install rfdetr
!pip install yt-dlp
!pip install -U ultralytics

In [ ]:
import os
import subprocess # Add subprocess module for better command execution control

print("📦 필요한 패키지를 확인/설치 중입니다...")
os.system("pip install -q yt-dlp")

#youtube_url = "https://www.youtube.com/watch?v=lGDjYTwh3MQ"
original_video_path = "/content/제일 처음 영상_23분.mp4"
input_video_path = "/content/input_video.mp4"

'''
print("\n📥 1. 유튜브 원본 영상 다운로드 중...")
# Use subprocess.run to capture stderr for better debugging
command = f"yt-dlp -f 'best[ext=mp4]' -o '{original_video_path}' {youtube_url}"
process = subprocess.run(command, shell=True, capture_output=True, text=True)

# Check for download failure using the return code
if process.returncode != 0:
    print(f"yt-dlp stdout:\n{process.stdout}")
    print(f"yt-dlp stderr:\n{process.stderr}")
    raise RuntimeError(f"❌ 다운로드에 실패했습니다! (에러 코드: {process.returncode})\nyt-dlp 에러 출력: {process.stderr}\nURL을 확인하거나 Colab 환경 문제일 수 있습니다.")

# Also check if the file actually exists, as yt-dlp might report success but fail to create file
if not os.path.exists(original_video_path):
    raise RuntimeError("❌ 다운로드에 실패했습니다! 영상 파일이 생성되지 않았습니다. URL을 확인하거나 Colab 환경 문제일 수 있습니다.")

print(f"✅ 원본 다운로드 성공! (파일크기: {os.path.getsize(original_video_path) / (1024*1024):.1f} MB)")
'''
print("✂️ 2. 영상 구간(8분 ~ 13분) 자르기 작업 중...")
# For ffmpeg, redirect stderr to /dev/null as it often prints verbose non-error info to stderr
cut_status = os.system(f"ffmpeg -y -i '{original_video_path}' -ss 00:08:00 -to 00:13:00 -c copy '{input_video_path}' 2>/dev/null")

# 자르기 실패 시 역시 코드를 정지시킵니다.
if cut_status != 0 or not os.path.exists(input_video_path):
    raise RuntimeError("❌ 영상 자르기에 실패했습니다!")

print(f"✅ 최종 영상 준비 완료! (파일크기: {os.path.getsize(input_video_path) / (1024*1024):.1f} MB)")
print("🎉 완벽합니다! 이제 아래의 AI 분석 셀을 실행하세요.")

## VIT를 위한


In [ ]:
import os
import random
from PIL import Image, ImageEnhance, ImageFilter

# ==========================================
# 1. 폴더 경로 및 생성할 이미지 개수 설정
# ==========================================
fg_dir = "/content/fg_image"                 # 투명한 미체결 후크(PNG) 폴더
bg_dir = "/content/bg_image"                 # 밧줄/배경 이미지 폴더
output_dir = "/content/synthetic_unconnected" # 완성본이 저장될 폴더
num_to_generate = 500                         # 뻥튀기할 횟수

os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 2. 🎨 핵심: 전경(후크)의 질감과 조명을 속이는 함수
# ==========================================
def augment_foreground(img):
    # 1) 70% 확률로 밝기 무작위 조절 (역광이나 쨍한 햇빛 모사)
    if random.random() < 0.7:
        enhancer = ImageEnhance.Brightness(img)
        img = enhancer.enhance(random.uniform(0.5, 1.5))

    # 2) 70% 확률로 대비 조절 (낡고 흐릿한 느낌 ~ 뚜렷한 느낌)
    if random.random() < 0.7:
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(random.uniform(0.5, 1.5))

    # 3) 70% 확률로 채도(색감) 조절 (빛바랜 후크 모사)
    if random.random() < 0.7:
        enhancer = ImageEnhance.Color(img)
        img = enhancer.enhance(random.uniform(0.3, 1.5))

    # 4) 50% 확률로 미세한 가우시안 흐림(Blur) 추가 (초점 나감 모사)
    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

    return img

# ==========================================
# 3. 자동 합성 파이프라인 실행
# ==========================================
def generate_robust_synthetic_data():
    fg_paths = [os.path.join(fg_dir, f) for f in os.listdir(fg_dir) if f.lower().endswith('.png')]
    bg_paths = [os.path.join(bg_dir, f) for f in os.listdir(bg_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if not fg_paths or not bg_paths:
        print("❌ 전경(투명 PNG) 또는 배경 이미지가 없습니다. 폴더와 파일을 확인해주세요!")
        return

    print(f"🚀 [강력 증강 모드] 총 {num_to_generate}장의 악질 합성 데이터 생성을 시작합니다...")

    for i in range(num_to_generate):
        bg = Image.open(random.choice(bg_paths)).convert("RGBA")
        fg = Image.open(random.choice(fg_paths)).convert("RGBA")

        # 💡 업그레이드 포인트: 합성 전에 질감과 조명 강제 변형!
        fg = augment_foreground(fg)

        # 크기 변형 (30% ~ 70% 찌그러뜨리기)
        scale = random.uniform(0.3, 0.7)
        new_w, new_h = int(fg.width * scale), int(fg.height * scale)
        fg = fg.resize((new_w, new_h), Image.Resampling.LANCZOS)

        # 각도 변형 (-45도 ~ 45도)
        angle = random.uniform(-45, 45)
        fg = fg.rotate(angle, expand=True, resample=Image.Resampling.BICUBIC)

        # 위치 무작위 계산
        max_x = max(0, bg.width - fg.width)
        max_y = max(0, bg.height - fg.height)
        paste_x, paste_y = random.randint(0, max_x), random.randint(0, max_y)

        # 투명도(Alpha)를 마스크로 삼아 완벽하게 배경에 안착
        bg.paste(fg, (paste_x, paste_y), fg)

        # JPG 변환 및 저장
        final_img = bg.convert("RGB")
        out_path = os.path.join(output_dir, f"robust_overlap_{i:04d}.jpg")
        final_img.save(out_path, quality=95)

    print(f"✅ 생성 완료! '{output_dir}' 폴더에 5장의 한계를 극복한 {num_to_generate}장의 데이터가 마련되었습니다.")

# 함수 실행
generate_robust_synthetic_data()

In [ ]:
!unzip "/content/합본데이터.zip" -d "/content/full"

In [ ]:
import os
import shutil

# 사진에 보이는 경로에 맞게 설정
source_dir = "/content/synthetic_unconnected"
destination_dir = "/content/full/합본데이터/unconnected"

if not os.path.exists(destination_dir):
    print(f"❌ '{destination_dir}' 경로를 찾을 수 없습니다. 폴더 이름이나 위치를 다시 확인해주세요.")
else:
    moved_count = 0
    # synthetic_unconnected 폴더 안의 모든 파일을 하나씩 꺼내서 이동
    for filename in os.listdir(source_dir):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        # 파일인 경우에만 덮어쓰기/이동 진행
        if os.path.isfile(source_file):
            shutil.move(source_file, destination_file)
            moved_count += 1

    print(f"✅ 완벽합니다! 총 {moved_count}장의 합성 데이터가 기존 '합본데이터/unconnected' 폴더로 쏙 들어갔습니다.")

In [ ]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. 경로 설정 (합성 데이터가 합쳐진 폴더)
dataset_dir = "/content/full/합본데이터"
connected_dir = os.path.join(dataset_dir, "connected")
unconnected_dir = os.path.join(dataset_dir, "unconnected")

# 2. 이미지 파일 싹 다 수집
connected_paths = [os.path.join(connected_dir, f) for f in os.listdir(connected_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
unconnected_paths = [os.path.join(unconnected_dir, f) for f in os.listdir(unconnected_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"📊 수집된 데이터 -> 체결(Connected): {len(connected_paths)}장, 미체결(Unconnected): {len(unconnected_paths)}장")

# 3. Train (80%) / Val (20%) 분할
random.seed(42)
random.shuffle(connected_paths)
random.shuffle(unconnected_paths)

c_split = int(len(connected_paths) * 0.8)
u_split = int(len(unconnected_paths) * 0.8)

c_train = connected_paths[:c_split]
c_val = connected_paths[c_split:]

u_train = unconnected_paths[:u_split]
u_val = unconnected_paths[u_split:]

# 4. 리스트 합치기 및 라벨링 (Connected: 0, Unconnected: 1)
train_samples = [(p, 0) for p in c_train] + [(p, 1) for p in u_train]
val_samples = [(p, 0) for p in c_val] + [(p, 1) for p in u_val]
random.shuffle(train_samples) # Train 데이터만 순서 섞기

# ==========================================
# 5. 데이터 증강 및 DataLoader(파이토치 밥그릇) 구성
# ==========================================
# ViT는 224x224 사이즈의 이미지를 16x16 조각(Patch)으로 잘라서 봅니다.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class HookDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# 🔥 에러가 났던 train_loader가 드디어 여기서 생성됩니다!
train_loader = DataLoader(HookDataset(train_samples, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(HookDataset(val_samples, val_transform), batch_size=32, shuffle=False)

print(f"✅ DataLoader 준비 완료! (Train: {len(train_samples)}장, Val: {len(val_samples)}장)")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class DissectedViT(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. 사전 학습된 ViT-Base 모델 불러오기
        self.vit = models.vit_b_16(weights='IMAGENET1K_V1')

        # 2. 🧊 데이터 부족(과적합) 방지를 위한 극단적 동결 (Freeze)
        # 모든 파라미터의 학습을 멈춤(얼림)
        for param in self.vit.parameters():
            param.requires_grad = False

        # 맨 마지막 Transformer 블록(11번째)만 녹여서 우리 데이터로 미세조정(Fine-Tuning)
        for param in self.vit.encoder.layers[-1].parameters():
            param.requires_grad = True

        # 3. 🔥 [강사님 요구사항] 14x14 Patch Feature Map용 공간 어텐션
        # ViT의 출력 차원은 768입니다. 768채널을 1개로 압축하여 집중할 곳을 찾습니다.
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(768, 1, kernel_size=1),
            nn.Sigmoid()
        )

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # 4. 최종 분류기 머리 (CLS 토큰 정보 + Patch 어텐션 정보 결합)
        self.classifier = nn.Sequential(
            nn.Linear(768 * 2, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(p=0.4),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        # --- [ViT 내부 동작 가로채기] ---
        # 1. 이미지를 Patch로 쪼개고 위치 정보 부여
        x = self.vit._process_input(x)
        n = x.shape[0]
        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)

        # 2. Transformer 인코더 통과 (Shape: [Batch, 197, 768])
        x = self.vit.encoder(x)

        # 3. 전체 요약본인 CLS 토큰과 196개의 Patch 토큰 분리
        cls_token = x[:, 0]        # [Batch, 768]
        patch_tokens = x[:, 1:]    # [Batch, 196, 768]

        # 4. 🔥 [핵심] 196개의 일렬로 선 Patch를 14x14 Feature Map으로 재조립!
        B, N, C = patch_tokens.shape
        # (Batch, Channel, Height, Width) 형태로 변환 -> [Batch, 768, 14, 14]
        spatial_features = patch_tokens.transpose(1, 2).view(B, C, 14, 14)

        # 5. 공간 어텐션 적용 (오버랩 된 교차점에 집중!)
        attention_mask = self.spatial_attention(spatial_features) # [Batch, 1, 14, 14]
        attended_features = spatial_features * attention_mask

        # 6. 어텐션이 적용된 특성지도를 하나로 압축
        pooled_features = self.avg_pool(attended_features).flatten(1) # [Batch, 768]

        # 7. CLS 토큰과 공간 정보를 합쳐서 최종 분류
        combined_features = torch.cat([cls_token, pooled_features], dim=1) # [Batch, 1536]

        return self.classifier(combined_features)

# 모델 생성
cls_model = DissectedViT()
print("✅ 강사님 맞춤형 [Feature Map 개조 + 극단적 동결] ViT 구조가 준비되었습니다!")

In [ ]:
import time
import torch
import torch.nn as nn

# 1. GPU 장치 설정 및 모델 이동
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cls_model = cls_model.to(device)

# ==========================================
# 💡 방금 말씀드린 수정 부분 (Loss 가중치 변경)
# ==========================================
# 데이터 비율(체결 1539 : 미체결 659)을 고려하여 미체결 오분류 시 벌점을 2.33배 부여!
loss_weights = torch.tensor([1.0, 2.33]).to(device)
criterion = nn.CrossEntropyLoss(weight=loss_weights)

# ViT는 학습률을 조금 더 낮게(안전하게) 가져가는 것이 좋습니다.
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, cls_model.parameters()), lr=5e-5, weight_decay=1e-4)
# ==========================================

# 3. 학습 루프 실행
num_epochs = 10
best_val_acc = 0.0

print(f"🚀 {device} 디바이스에서 [ViT Feature Map 개조 버전] 학습을 시작합니다.")

for epoch in range(num_epochs):
    # --- 훈련 모드 ---
    cls_model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cls_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = (train_correct / train_total) * 100

    # --- 검증 모드 ---
    cls_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cls_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = (val_correct / val_total) * 100

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% || "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

    # 최고 가중치 저장 (이번엔 이름에 vit를 넣어서 구분할게요)
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        save_path = '/content/best_custom_hook_vit.pth'
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': cls_model.state_dict(),
            'val_acc': best_val_acc,
        }, save_path)
        print(f"   ⭐ 최고 성능 갱신! ViT 모델 가중치가 저장되었습니다. (정확도: {best_val_acc:.2f}%)")

print("\n🎉 모든 학습 순서가 안전하게 끝났습니다! 저장된 'best_custom_hook_vit.pth' 파일을 테스트에 사용하시면 됩니다.")

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정 (코랩에서 폰트 깨짐 방지용)
plt.rc('font', family='NanumGothic')

# 1. 디바이스 및 모델 준비
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# (주의) 만약 런타임이 초기화되었다면 DissectedViT 클래스가 선언된 셀을 먼저 실행해야 합니다.
test_model = DissectedViT().to(device)

# 2. 학습된 가중치 로드
weight_path = '/content/best_custom_hook_vit.pth'
try:
    checkpoint = torch.load(weight_path, map_location=device)
    test_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ 성공: '{weight_path}' 가중치를 성공적으로 로드했습니다! (당시 검증 정확도: {checkpoint['val_acc']:.2f}%)")
except Exception as e:
    print(f"❌ 가중치 로드 실패: {e}")

test_model.eval()

# 3. 테스트할 이미지 전처리 설정 (검증 데이터와 동일하게 224x224 적용)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==========================================
# 🎯 테스트할 이미지 경로 설정
# ==========================================
# 테스트하고 싶은 이미지의 경로를 넣어주세요. (예: "안전고리 겹치는.png")
test_image_path = "/content/0063.png"

def predict_single_image(image_path, model, transform, device):
    try:
        # 이미지 열기 및 RGB 변환
        original_img = Image.open(image_path).convert('RGB')

        # 전처리 및 배치 차원 추가 (1, C, H, W)
        input_tensor = transform(original_img).unsqueeze(0).to(device)

        # 모델 예측
        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = F.softmax(outputs, dim=1)[0] # 확률값 계산
            confidence, predicted_class = torch.max(probabilities, 0)

        # 라벨 매핑 (0: 체결, 1: 미체결)
        class_names = ['체결 (Connected)', '미체결 (Unconnected)']
        pred_label = class_names[predicted_class.item()]
        conf_score = confidence.item() * 100

        # 결과 출력 및 시각화
        print("\n" + "="*40)
        print(f"🔍 테스트 이미지: {image_path.split('/')[-1]}")
        print(f"🤖 모델 예측 결과: {pred_label}")
        print(f"📈 예측 신뢰도(확률): {conf_score:.2f}%")
        print("="*40 + "\n")

        plt.figure(figsize=(6, 6))
        plt.imshow(original_img)
        plt.title(f"예측: {pred_label} ({conf_score:.2f}%)", fontsize=16, color='red' if predicted_class.item() == 1 else 'blue')
        plt.axis('off')
        plt.show()

    except FileNotFoundError:
        print(f"❌ '{image_path}' 파일을 찾을 수 없습니다. 경로와 파일명을 다시 확인해주세요.")

# 함수 실행
predict_single_image(test_image_path, test_model, test_transform, device)

VIT 다시

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import random

# ==========================================
# 1. 실제 크롭 데이터 경로 설정 (수정 필요)
# ==========================================
# (예시) 체결 1539장, 미체결 459장이 들어있는 폴더 경로
connected_dir = "/content/dataset/connected"
unconnected_dir = "/content/dataset/unconnected"

c_paths = [os.path.join(connected_dir, f) for f in os.listdir(connected_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
u_paths = [os.path.join(unconnected_dir, f) for f in os.listdir(unconnected_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"📊 수집된 찐 데이터 -> 체결: {len(c_paths)}장, 미체결: {len(u_paths)}장")

# Train (80%) / Val (20%) 분할
random.seed(42)
random.shuffle(c_paths)
random.shuffle(u_paths)

c_train, c_val = c_paths[:int(len(c_paths)*0.8)], c_paths[int(len(c_paths)*0.8):]
u_train, u_val = u_paths[:int(len(u_paths)*0.8)], u_paths[int(len(u_paths)*0.8):]

train_samples = [(p, 0) for p in c_train] + [(p, 1) for p in u_train]
val_samples = [(p, 0) for p in c_val] + [(p, 1) for p in u_val]
random.shuffle(train_samples)

# ==========================================
# 2. 순정 ViT를 위한 표준 전처리
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10), # 크롭 이미지이므로 회전은 너무 크지 않게
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class PureHookDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_loader = DataLoader(PureHookDataset(train_samples, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(PureHookDataset(val_samples, val_transform), batch_size=32, shuffle=False)

# byte tracking 및 smoothing 활용

In [ ]:
pip install --upgrade supervision

In [ ]:
!pip install -q supervision
!pip install rfdetr

In [ ]:
!pip install -U ultralytics

In [ ]:
import cv2
import numpy as np
import supervision as sv
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from rfdetr import RFDETRSegLarge
from collections import deque

input_video_path = "/content/input_video.mp4"
output_video_path = "/content/output2_video.mp4"

# ==========================================
# 1. AI 모델 로드
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 디바이스: {device}")

print("📥 1단계: RF-DETR (Segmentation) 모델 로드 중...")
model = RFDETRSegLarge(pretrain_weights="/content/checkpoint_best_ema (2).pth")

print("📥 2단계: 개조된 EfficientNet-B0 (Classification) 모델 로드 중...")

# 💡 [필수 추가] 우리가 만든 커스텀 풀링 클래스 선언
class CustomPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):
        return torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1)

# 모델 뼈대 선언 및 수술 진행
cls_model = models.efficientnet_b0(weights=None)
cls_model.avgpool = CustomPooling()

in_features = 1280 * 2
cls_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(512, 2)
)

# 💡 파일명 확인: 아까 저장한 커스텀 모델의 이름으로 지정!
efficientnet_path = "/content/best_custom_hook_classifier.pth"

checkpoint = torch.load(efficientnet_path, map_location=device)
cls_model.load_state_dict(checkpoint['model_state_dict'])
cls_model = cls_model.to(device)
cls_model.eval()

# (이하 cls_transform 부분은 기존과 동일하게 유지)
cls_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==========================================
# 2. 클래스 맵 및 설정
# ==========================================
RF_DETR_CLASSES = {
    1: "harness",
    2: "hook",
    3: "lanyard",
    4: "lifeline",
    5: "worker"
}
HOOK_CLASS_IDS = [2]
WORKER_CLASS_IDS = [5]

MARGIN_RATIO = 0.5
STRICT_THRESHOLD = 0.85

tracker = sv.ByteTrack()
smoother = sv.DetectionsSmoother()
HISTORY_LENGTH = 15
hook_history = {}

mask_annotator = sv.MaskAnnotator()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

# ==========================================
# 3. 영상 프레임별 처리 루프 (OpenCV 표준 방식으로 전면 교체)
# ==========================================
cap = cv2.VideoCapture(input_video_path)

# 원본 비디오로부터 다이렉트로 정보 추출 (오류 방지)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("\n======= 📊 원본 영상 정보 확인 =======")
print(f" 가로 크기: {width} px")
print(f" 세로 크기: {height} px")
print(f" 프레임 수(FPS): {fps}")
print(f" 총 프레임 개수: {total_frames} 개")
print("=======================================\n")

# 예외 처리: 비디오 파일이 열리지 않거나 정보가 깨진 경우
if width == 0 or height == 0 or total_frames == 0:
    raise ValueError(f"❌ 영상을 읽어오지 못했습니다! 파일 경로를 다시 확인해주세요: {input_video_path}")

# 가장 범용적이고 안전한 mp4v 코덱 사용 명시
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print("🎬 영상 처리를 시작합니다...")
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break # 영상 피드가 끝나면 자동 종료

    frame_idx += 1
    # 30프레임마다 콘솔에 진행 상황 표시 (멈춤 현상 모니터링용)
    if frame_idx % 30 == 0 or frame_idx == total_frames:
        print(f"⏳ 영상 처리 중... [{frame_idx}/{total_frames}] ({(frame_idx/total_frames)*100:.1f}%)")

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)

    # 1. 객체 탐지 (임계값 0.4로 완화하여 박스 깜빡임 방지)
    detections = model.predict(images=pil_image, threshold=0.4)

    # 메타데이터 초기화 (Smoother 에러 원인 제거)
    if hasattr(detections, 'metadata') and isinstance(detections.metadata, dict):
        detections.metadata = {}

    # 2. ByteTrack 추적 및 Smoother 적용
    detections = tracker.update_with_detections(detections)
    detections = smoother.update_with_detections(detections)

    custom_labels = []
    hook_alerts = []

    # 현재 프레임의 작업자(Worker) 정보 수집
    active_workers = []
    if detections.tracker_id is not None:
        for i in range(len(detections.xyxy)):
            if detections.class_id[i] in WORKER_CLASS_IDS:
                w_xmin, w_ymin, w_xmax, w_ymax = map(int, detections.xyxy[i])
                w_id = detections.tracker_id[i]
                active_workers.append({
                    'id': w_id,
                    'bbox': (w_xmin, w_ymin, w_xmax, w_ymax),
                    'top_center': (w_xmin + (w_xmax - w_xmin) // 2, w_ymin)
                })

    if detections.tracker_id is not None:
        for i in range(len(detections.xyxy)):
            xyxy = detections.xyxy[i]
            class_id = detections.class_id[i]
            conf = detections.confidence[i]
            tracker_id = detections.tracker_id[i]

            class_name = RF_DETR_CLASSES.get(class_id, f"Class {class_id}")

            if class_id in HOOK_CLASS_IDS:
                xmin, ymin, xmax, ymax = map(int, xyxy)

                # 후크의 중심점 계산
                hook_cx = xmin + (xmax - xmin) // 2
                hook_cy = ymin + (ymax - ymin) // 2

                # 기하학적 매칭 (어느 작업자의 박스 내부에 포함되는가?)
                matched_worker = None
                for worker in active_workers:
                    wx1, wy1, wx2, wy2 = worker['bbox']
                    if wx1 <= hook_cx <= wx2 and wy1 <= hook_cy <= wy2:
                        matched_worker = worker
                        break

                # 후크 인근 크롭 및 EfficientNet 체결 판별
                hook_w, hook_h = xmax - xmin, ymax - ymin
                margin_px = int(max(hook_w, hook_h) * MARGIN_RATIO)

                c_xmin = max(0, xmin - margin_px)
                c_ymin = max(0, ymin - margin_px)
                c_xmax = min(frame.shape[1], xmax + margin_px)
                c_ymax = min(frame.shape[0], ymax + margin_px)

                cropped_img = frame_rgb[c_ymin:c_ymax, c_xmin:c_xmax]

                if cropped_img.size == 0:
                    custom_labels.append(f"Hook:{tracker_id} {conf:.2f}")
                    continue

                pil_crop = Image.fromarray(cropped_img)
                input_tensor = cls_transform(pil_crop).unsqueeze(0).to(device)

                with torch.no_grad():
                    outputs = cls_model(input_tensor)
                    probs = F.softmax(outputs, dim=1)

                prob_connected = probs[0][0].item()

                if tracker_id not in hook_history:
                    hook_history[tracker_id] = deque(maxlen=HISTORY_LENGTH)
                hook_history[tracker_id].append(prob_connected)

                avg_prob_connected = sum(hook_history[tracker_id]) / len(hook_history[tracker_id])

                if avg_prob_connected >= STRICT_THRESHOLD:
                    status = 'connected'
                    score = avg_prob_connected * 100
                else:
                    status = 'unconnected'
                    score = (1 - avg_prob_connected) * 100

                if status == 'connected':
                    custom_labels.append(f"H:{tracker_id} 🟢 SAFE")
                else:
                    custom_labels.append(f"H:{tracker_id} 🚨 DANGER")

                hook_alerts.append((xmin, ymin, xmax, ymax, status, score, tracker_id, matched_worker))

            else:
                if class_id in WORKER_CLASS_IDS:
                    custom_labels.append(f"Worker:{tracker_id}")
                else:
                    custom_labels.append(f"ID:{tracker_id} {class_name}")

    # 마스킹 및 기본 바운딩 박스 시각화
    annotated_frame = mask_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=custom_labels)

    # 커스텀 선 연결 및 경고 박스 렌더링
    for xmin, ymin, xmax, ymax, status, score, h_id, worker in hook_alerts:
        color = (0, 255, 0) if status == 'connected' else (0, 0, 255)

        if worker:
            alert_text = f"[ W:{worker['id']} - SAFE ]" if status == 'connected' else f"[ W:{worker['id']} - DANGER ]"
            hook_cx = xmin + (xmax - xmin) // 2
            hook_cy = ymin + (ymax - ymin) // 2
            cv2.line(annotated_frame, worker['top_center'], (hook_cx, hook_cy), color, 2, cv2.LINE_AA)
        else:
            alert_text = f"[ UNKNOWN W - SAFE ]" if status == 'connected' else f"[ UNKNOWN W - DANGER ]"

        cv2.rectangle(annotated_frame, (xmin, ymin), (xmax, ymax), color, 4)
        cv2.putText(annotated_frame, alert_text, (xmin, max(30, ymin - 35)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

    # 파일에 실시간 프레임 데이터 입력
    out.write(annotated_frame)

# 프로세스 종료 후 스트림 해제 (파일 깨짐 방지 락 해제)
cap.release()
out.release()

print(f"\n🎉 영상 처리 완전히 완료! 결과 파일 용량을 확인해보세요.")

# efficient net 좀 뜯어서 학습

In [ ]:
import os
import zipfile
import glob
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

# ==========================================
# 0. 파일 압축 해제 및 경로 설정
# ==========================================
zip_path = "/content/합본데이터.zip"
extract_path = "/content/dataset"

if os.path.exists(zip_path):
    print("📂 1단계: 압축 파일 해제 중...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("🔓 압축 해제 완료!")
else:
    print("❌ '/content/합본데이터.zip' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

# 하위 폴더 자동 탐색 (폴더 구조 유연성 확보)
connected_all = glob.glob(os.path.join(extract_path, "**", "connected", "*.*"), recursive=True)
unconnected_all = glob.glob(os.path.join(extract_path, "**", "unconnected", "*.*"), recursive=True)

print(f"📊 원본 데이터 개수 -> Connected: {len(connected_all)}장, Unconnected: {len(unconnected_all)}장")

# ==========================================
# 1. 영리한 데이터 밸런싱 (Train/Val 분리 후 조절)
# ==========================================
random.seed(42)
random.shuffle(connected_all)
random.shuffle(unconnected_all)

# 80%는 학습(Train), 20%는 검증(Val)으로 기본 분할
c_split = int(len(connected_all) * 0.8)
u_split = int(len(unconnected_all) * 0.8)

c_train_orig = connected_all[:c_split]
c_val = connected_all[c_split:]

u_train_orig = unconnected_all[:u_split]
u_val = unconnected_all[u_split:]

# 🔥 [핵심 타협점] 학습 데이터셋 밸런싱 진행
# 1. Connected 축소: 약 1200장 중 800장만 랜덤 추출
c_train_balanced = random.sample(c_train_orig, min(800, len(c_train_orig)))

# 2. Unconnected 증강: 160장 목록을 2.5배 뻥튀기하여 400장으로 채움
# (경로 리스트만 복사해두면, PyTorch가 읽을 때 실시간으로 무작위 증강을 적용하므로 용량을 차지하지 않음)
target_u_count = 400
import random # 만약을 대비해 다시 임포트
u_train_balanced = random.choices(u_train_orig, k=target_u_count)

# 최종 학습 리스트 구성 (이미지 경로, 라벨[0:connected, 1:unconnected])
train_samples = [(p, 0) for p in c_train_balanced] + [(p, 1) for p in u_train_balanced]
val_samples = [(p, 0) for p in c_val] + [(p, 1) for p in u_val]

random.shuffle(train_samples) # 데이터 섞기

print(f"⚖️ 밸런싱 후 학습 데이터 -> Connected: {len(c_train_balanced)}장, Unconnected: {len(u_train_balanced)}장 (총 {len(train_samples)}장)")
print(f"🧪 검증 데이터 (원본 비율 유지) -> Connected: {len(c_val)}장, Unconnected: {len(u_val)}장 (총 {len(val_samples)}장)")

# ==========================================
# 2. 데이터셋 클래스 및 데이터 증강(Augmentation) 정의
# ==========================================
# 후크의 특성상 상하 반전(VerticalFlip)은 중력 방향을 파괴하므로 제외하고, 미세 회전과 밝기 조절만 적용
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class HookDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_loader = DataLoader(HookDataset(train_samples, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(HookDataset(val_samples, val_transform), batch_size=32, shuffle=False)

# ==========================================
# 3. 모델 내부 구조 개조 (Custom Pooling + Multi-Layer Classifier)
# ==========================================
class CustomPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):
        # 평균 피처와 최댓값 피처를 가로로 결합(Concat)하여 미세한 걸쇠 특징을 보존
        return torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1)

# 순정 EfficientNet-B0 로드
cls_model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# 🛠️ 뼈대 수술 시작
cls_model.avgpool = CustomPooling() # 순정 GAP 제거 후 커스텀 풀링 이식
in_features = 1280 * 2              # 두 개의 풀링을 합쳤으므로 입력 차원이 2倍가 됨 (2560)

# 순정 단층 Classifier를 다층 MLP 구조로 전면 교체
cls_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(512, 2)
)

# ==========================================
# 4. 🔍 강사님 제출용: 모델 내부 구조 뜯어보기(분석) 코드
# ==========================================
print("\n=======================================================")
print("🔍 [MODEL DISSECTION] 개조된 EfficientNet-B0 구조 분석")
print("=======================================================")
print(f"1. 변경된 Pooling 레이어: {cls_model.avgpool}")
print(f"   -> 원리: 미세 걸쇠의 뾰족한 특징(Max)과 전체 형태(Avg)를 동시에 추출하도록 개조됨.")
print("\n2. 변경된 Classifier 레이어 구조:")
for i, layer in enumerate(cls_model.classifier):
    print(f"   [{i}] Layer: {layer}")
print("   -> 원리: 단순 선형 결합에서 비선형성(SiLU)과 과적합 방지(Dropout, BatchNorm)가 설계된 MLP로 고도화됨.")

# 레이어별 파라미터 학습 가능 여부 체크 및 총 파라미터 수 계산
total_params = sum(p.numel() for p in cls_model.parameters())
trainable_params = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
print(f"\n3. 파라미터 통계:")
print(f"   - 총 파라미터 수: {total_params:,} 개")
print(f"   - 학습 가능한 파라미터 수: {trainable_params:,} 개")
print("=======================================================\n")

# ==========================================
# 5. 불균형 대응 손실 함수(Loss) 및 학습 세팅
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cls_model = cls_model.to(device)

# 최종 가중치 부여 (2:1 비율에 맞춰 미체결을 틀렸을 때 벌점을 2배로 부여)
loss_weights = torch.tensor([1.0, 2.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=loss_weights)
optimizer = torch.optim.AdamW(cls_model.parameters(), lr=1e-4, weight_decay=1e-4)

print(f"🚀 {device} 디바이스에서 모든 학습 준비가 완료되었습니다! 이제 학습 루프를 실행하시면 됩니다.")

# efficient net

In [ ]:
!unzip -qq "/content/합본데이터 (2).zip" -d "/content/full"

In [ ]:
import os
import random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch
import torch.nn as nn
from PIL import Image

# ==========================================
# 0. 폴더 구조에 맞춘 안전한 데이터 탐색
# ==========================================
extract_path = "/content/full/합본데이터"

connected_all = []
unconnected_all = []

# 허용할 이미지 확장자
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            full_path = os.path.join(root, file)
            if "unconnected" in root.lower():
                unconnected_all.append(full_path)
            elif "connected" in root.lower():
                connected_all.append(full_path)

print("📂 하위 폴더 탐색 완료!")
print(f"📊 수집된 원본 데이터 -> Connected: {len(connected_all)}장, Unconnected: {len(unconnected_all)}장")

# ==========================================
# 1. 8:1:1 데이터 분할 및 밸런싱 (수정됨)
# ==========================================
random.seed(42)
random.shuffle(connected_all)
random.shuffle(unconnected_all)

# 🔥 80%(Train), 10%(Val), 10%(Test) 분할 인덱스 계산
c_train_idx = int(len(connected_all) * 0.8)
c_val_idx = int(len(connected_all) * 0.9)

u_train_idx = int(len(unconnected_all) * 0.8)
u_val_idx = int(len(unconnected_all) * 0.9)

# Connected 분할
c_train_orig = connected_all[:c_train_idx]
c_val = connected_all[c_train_idx:c_val_idx]
c_test = connected_all[c_val_idx:]

# Unconnected 분할
u_train_orig = unconnected_all[:u_train_idx]
u_val = unconnected_all[u_train_idx:u_val_idx]
u_test = unconnected_all[u_val_idx:]

# 🔥 학습 데이터셋 밸런싱 (Train 데이터만 밸런싱하고, Val/Test는 실제 환경 비율 유지)
target_c_count = min(800, len(c_train_orig))
c_train_balanced = random.sample(c_train_orig, target_c_count)

target_u_count = 400
u_train_balanced = random.choices(u_train_orig, k=target_u_count)

# 최종 리스트 구성 (라벨 0: Connected, 1: Unconnected)
train_samples = [(p, 0) for p in c_train_balanced] + [(p, 1) for p in u_train_balanced]
val_samples = [(p, 0) for p in c_val] + [(p, 1) for p in u_val]
test_samples = [(p, 0) for p in c_test] + [(p, 1) for p in u_test]

random.shuffle(train_samples)

print("\n==========================================")
print(f"⚖️ Train 데이터 (학습용) -> Connected: {len(c_train_balanced)}장, Unconnected: {len(u_train_balanced)}장 (총 {len(train_samples)}장)")
print(f"🧪 Val 데이터 (검증용) -> Connected: {len(c_val)}장, Unconnected: {len(u_val)}장 (총 {len(val_samples)}장)")
print(f"🎯 Test 데이터 (최종 평가용) -> Connected: {len(c_test)}장, Unconnected: {len(u_test)}장 (총 {len(test_samples)}장)")
print("==========================================")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import time

# ==========================================
# 2. 데이터셋 클래스 및 이미지 증강(Augmentation) 정의
# ==========================================
# 후크 이미지 특성상 상하 반전은 중력 방향을 파괴하므로 제외, 미세 회전과 밝기 조절만 적용
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class HookDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# 이전 셀에서 만든 train_samples, val_samples 활용
train_loader = DataLoader(HookDataset(train_samples, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(HookDataset(val_samples, val_transform), batch_size=32, shuffle=False)


# ==========================================
# 3. 🔥 [진짜 모델 구조 뜯어고치기] (Custom Pooling + MLP Classifier)
# ==========================================
class CustomPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):
        # 💡 [구조 개조 1] 순정 GAP의 단점을 보완하기 위해 평균값과 최댓값 특징을 가로로 결합(Concat)
        return torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1)

# 순정 EfficientNet-B0 모델 로드
cls_model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# 🛠️ 수술 1: 원래 탑재된 순정 풀링 레이어(avgpool)를 박살내고 우리가 만든 커스텀 풀링으로 이식
cls_model.avgpool = CustomPooling()
in_features = 1280 * 2  # 두 개의 풀링을 합쳤으므로 입력 차원이 2배(2560)로 확장됨

# 🛠️ 수술 2: 단순 선형 층 1개짜리 순정 분류기 머리를 깊은 다층 퍼셉트론(MLP) 구조로 전면 교체
cls_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),         # 과적합 방지 1
    nn.Linear(in_features, 512),             # 은닉층 추가 (2560 -> 512 압축)
    nn.BatchNorm1d(512),                     # 학습 안정화
    nn.SiLU(),                               # EfficientNet 최적화 활성화 함수
    nn.Dropout(p=0.3, inplace=True),         # 과적합 방지 2
    nn.Linear(512, 2)                        # 최종 출력 (Connected, Unconnected 2개)
)


# ==========================================
# 4. 🔍 강사님 제출용: 모델 구조 개조 결과 검증 및 분석 출력
# ==========================================
print("\n=======================================================")
print("🔍 [MODEL DISSECTION] 개조된 EfficientNet-B0 구조 분석")
print("=======================================================")
print(f"1. 변경된 Pooling 레이어: {cls_model.avgpool}")
print(f"2. 변경된 Classifier 레이어 구조:")
for i, layer in enumerate(cls_model.classifier):
    print(f"   [{i}] Layer: {layer}")
print("=======================================================\n")


# ==========================================
# 5. 불균형 대응 손실 함수(Loss) 및 학습 세팅
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cls_model = cls_model.to(device)

# 최종 가중치 부여 (데이터 비율에 맞춰 미체결을 틀렸을 때 벌점을 2배로 부여)
loss_weights = torch.tensor([1.0, 2.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=loss_weights)
optimizer = torch.optim.AdamW(cls_model.parameters(), lr=1e-4, weight_decay=1e-4)


# ==========================================
# 6. 🏃‍♂️ [안전 최우선] Recall 기반 에포크(Epoch) 학습 루프
# ==========================================
import time

num_epochs = 15  # 꼼꼼한 학습을 위해 15 에포크로 소폭 상향
best_val_recall = 0.0 # 🔥 최고 정확도가 아닌 '최고 미체결 Recall'을 추적합니다.

print("🔥 [안전 최우선 모드] 미체결(Unconnected) 탐지에 집중하여 학습을 시작합니다!")
start_time = time.time()

for epoch in range(num_epochs):
    # --- [Train Mode] ---
    cls_model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cls_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = (train_correct / train_total) * 100

    # --- [Validation Mode] ---
    cls_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    # 🔥 미체결(Unconnected, 라벨 1) 추적을 위한 변수 추가
    val_unconnected_true = 0
    val_unconnected_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cls_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            # 🔥 미체결(1) 데이터만 쏙 뽑아서 얼마나 맞췄는지 계산
            val_unconnected_true += (labels == 1).sum().item()
            val_unconnected_correct += ((predicted == 1) & (labels == 1)).sum().item()

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = (val_correct / val_total) * 100

    # 🔥 미체결 Recall(재현율) 계산: 실제 미체결 중 모델이 미체결로 맞춘 비율
    if val_unconnected_true > 0:
        epoch_val_recall = (val_unconnected_correct / val_unconnected_true) * 100
    else:
        epoch_val_recall = 0.0

    # 결과 출력 (Recall 강조)
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {epoch_train_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
    print(f"   🚨 [안전 지표] 미체결 Recall: {epoch_val_recall:.2f}% ({val_unconnected_correct}/{val_unconnected_true}장 정확히 탐지)")

    # 🔥 저장 기준 변경: 전체 정확도가 떨어지더라도 미체결 Recall이 높으면 무조건 저장!
    if epoch_val_recall >= best_val_recall:
        best_val_recall = epoch_val_recall
        save_path = '/content/best_custom_hook_classifier.pth'
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': cls_model.state_dict(),
            'val_acc': epoch_val_acc,
            'val_recall': best_val_recall, # Recall 값도 함께 저장
        }, save_path)
        print(f"   ⭐ 최고 안전성 갱신! 모델이 저장되었습니다. (미체결 Recall: {best_val_recall:.2f}%)")
    print("-" * 60)

total_time = time.time() - start_time
print(f"\n🎉 모든 학습이 완료되었습니다! (소요 시간: {total_time/60:.2f}분)")

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# 1. Test 데이터 로더 준비 (이전 셀에서 만든 test_samples 활용)
# 만약 test_samples가 날아갔다면 8:1:1 분할 코드를 다시 한번 실행해주세요!
try:
    test_loader = DataLoader(HookDataset(test_samples, val_transform), batch_size=32, shuffle=False)
except NameError:
    print("❌ test_samples 또는 HookDataset이 정의되지 않았습니다. 데이터 분할 셀을 먼저 실행해주세요!")

# 2. 모델 평가 모드 확인
model.eval()

# 테스트 데이터 전체에 대한 모델의 원시 예측 확률을 먼저 싹 다 모아둡니다.
all_probs = []
all_labels = []

print("🔍 Test 데이터 전체에 대한 예측을 수행 중입니다...")
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        # Softmax로 확률 변환 후, '체결(0)'일 확률만 추출
        probs = F.softmax(outputs, dim=1)[:, 0].cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

# 3. 다양한 임계값(Threshold) 시뮬레이션
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

print("\n" + "="*70)
print(f"{'Threshold(기준)':<15} | {'미체결 탐지율 (Recall)':<20} | {'체결 오경보율 (False Alarm)':<20}")
print("="*70)

for t in thresholds:
    tp = 0 # True Positive: 실제 미체결을 미체결로 맞춤 (최고 중요)
    fn = 0 # False Negative: 실제 미체결인데 체결이라고 놓침 (대형 사고)
    fp = 0 # False Positive: 실제 체결인데 미체결이라고 알람 울림 (작업자 짜증)
    tn = 0 # True Negative: 실제 체결을 체결로 맞춤

    for prob_connected, true_label in zip(all_probs, all_labels):
        # 예측: 체결 확률이 임계값 이상이면 체결(0), 아니면 미체결(1)로 강제 지정
        pred_label = 0 if prob_connected >= t else 1

        if true_label == 1 and pred_label == 1:
            tp += 1
        elif true_label == 1 and pred_label == 0:
            fn += 1
        elif true_label == 0 and pred_label == 1:
            fp += 1
        elif true_label == 0 and pred_label == 0:
            tn += 1

    # 지표 계산
    recall = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0.0
    false_alarm = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0

    # 출력 포맷팅
    print(f"체결 기준 {t*100:2.0f}% 이상 | 🚨 미체결 탐지: {recall:5.1f}% | ⚠️ 오경보(짜증): {false_alarm:5.1f}%")

print("="*70)
print("💡 해석 가이드:")
print("- 미체결 탐지(Recall)는 무조건 높을수록(100%에 가까울수록) 안전합니다.")
print("- 단, 기준을 너무 높이면 멀쩡한 고리에도 알람이 울려 오경보(False Alarm)가 급증합니다.")
print("- 오경보가 10~15%를 넘지 않으면서 미체결 탐지율이 가장 높은 구간이 '최적의 타협점'입니다.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import os

# ==========================================
# 1. 테스트할 이미지 경로 설정
# ==========================================
# 💡 코랩 왼쪽 폴더에서 아무 이미지나 우클릭 -> '경로 복사' 후 여기에 붙여넣으세요!
test_image_path = "/content/KakaoTalk_20260710_144302840.png"

# ==========================================
# 2. 학습할 때와 똑같은 구조로 모델 뼈대 구축 (필수)
# ==========================================
class CustomPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):
        return torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1)

# 모델 뼈대 선언
model = models.efficientnet_b0(weights=None) # 학습된 가중치를 덮어씌울 것이므로 None
model.avgpool = CustomPooling()

in_features = 1280 * 2
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(512, 2)
)

# ==========================================
# 3. 저장된 최고 성능 가중치 파일(.pth) 불러오기
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = '/content/best_custom_hook_classifier.pth'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval() # ⚠️ 매우 중요: 추론 모드로 설정 (Dropout, BatchNorm 동결)
    print(f"✅ 성공: '{checkpoint_path}' 가중치를 성공적으로 로드했습니다! (당시 검증 정확도: {checkpoint['val_acc']:.2f}%)")
else:
    print(f"❌ 오류: '{checkpoint_path}' 파일을 찾을 수 없습니다. 학습이 완료되었는지 확인하세요.")

# 학습할 때 썼던 Val 전처리 가이드라인과 100% 일치해야 합니다.
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 💡 threshold 파라미터 추가 (기본값 0.8 = 80%)
def predict_hook(img_path, threshold=0.80):
    if not os.path.exists(img_path):
        print(f"❌ 이미지를 찾을 수 없습니다: {img_path}")
        return

    # 이미지 열기 및 전처리
    raw_img = Image.open(img_path).convert('RGB')
    input_tensor = inference_transform(raw_img).unsqueeze(0) # 배치 차원 추가 (1, 3, 224, 224)
    input_tensor = input_tensor.to(device)

    # 예측 진행
    with torch.no_grad():
        outputs = model(input_tensor)

        # Softmax를 거쳐 각 클래스별 확률(Probability) 계산
        probabilities = F.softmax(outputs, dim=1)[0]

    # ==========================================
    # 🔥 [핵심] 80% 임계값(Threshold) 판별 로직
    # ==========================================
    # 인덱스 0: 체결(Connected) 확률
    # 인덱스 1: 미체결(Unconnected) 확률
    prob_connected = probabilities[0].item()
    prob_unconnected = probabilities[1].item()

    # 체결 확률이 우리가 정한 threshold(80%) 이상일 때만 체결로 인정!
    if prob_connected >= threshold:
        predicted_idx = 0
        final_confidence = prob_connected
    else:
        # 체결 확률이 80% 미만이면 가차 없이 미체결로 튕겨냄
        predicted_idx = 1
        final_confidence = prob_unconnected

    # 결과 해석
    class_names = ["체결 (Connected)", "미체결 (Unconnected) ⚠️"]
    result_text = class_names[predicted_idx]
    percentage = final_confidence * 100

    print("\n" + "="*45)
    print(f"🔍 테스트 이미지: {os.path.basename(img_path)}")
    print(f"⚙️ 적용된 안전 기준(Threshold): {threshold * 100:.0f}%")
    print(f"🤖 모델 예측 결과: {result_text}")
    print(f"📈 최종 예측 신뢰도: {percentage:.2f}%")
    print("-" * 45)
    print(f"   [상세 확률] 체결: {prob_connected*100:.2f}% / 미체결: {prob_unconnected*100:.2f}%")
    print("="*45)

# 함수 실행 (원하는 Threshold 값을 마음대로 조절할 수 있습니다!)
predict_hook(test_image_path, threshold=0.80)